# 🧬 VegPep Pipeline v10 — Single-Filter Mode (Table S2)
## STOP after STEP 3 — produces the 6 standalone counts for the SPAFE paper

**Difference vs standard pipeline v9:**
- Standard pipeline: continues with ESM-2 mining, Boltz-2, gold candidates.
- This version: **stops after STEP 3 (hydrolysis)** and computes the 6 standalone counts for Table S2.


**For the SPAFE/MDM2 paper (Zingiber officinale → SPAFESTWDILK):** at STEP 0b select DATABASE=`B` (phytotherapeutics). The ESM-2 mining target does not affect STEP 3b counts (filters are target-independent).

**Run in order:**
1. STEP 0 — dependencies
2. STEP 0b — DB/target selection
3. STEP 0c — DB loading
4. STEP 1a / 1b — target / proteome configuration
5. STEP 2 — load proteomes
6. STEP 3 — enzymatic hydrolysis → generates `peptides_hydrolysis.csv`
7. **STEP 3b — Single-Filter Mode → final output for Minoo**

No ESM-2, no Boltz-2, no SQLite database.


In [1]:
# ============================================================
# STEP 0 — DEPENDENCY INSTALLATION
# ============================================================
# Pinned versions to avoid transformers/torch device_mesh
# incompatibility (transformers >=4.45 requires torch >=2.5).
import subprocess, sys

PINNED = [
    "transformers==4.44.2",
    "tokenizers>=0.19,<0.20",
]
OTHERS = [
    "biopython",
    "pandas",
    "matplotlib",
    "seaborn",
    "tqdm",
    "pyyaml",
]

print("\U0001F4E6 Installing dependencies...")

print("   \U0001F4E5 Installing transformers==4.44.2 + tokenizers (pinned)...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + PINNED
)

for pkg in OTHERS:
    mod = "yaml" if pkg == "pyyaml" else pkg.split("[")[0]
    try:
        __import__(mod)
        print(f"   \u2705 {pkg} already installed")
    except ImportError:
        print(f"   \U0001F4E5 Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"   \u2705 {pkg} installed")

import torch
print(f"\n   torch {torch.__version__}")
import transformers
print(f"   transformers {transformers.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n\U0001F5A5\uFE0F  GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("\n\u26A0\uFE0F  No GPU detected (not needed for single-filter mode).")

print("\n\u2705 Setup complete!")
print("\u26A0\uFE0F  If you just upgraded transformers, RESTART THE KERNEL before continuing.")


📦 Installing dependencies...
   📥 Installing transformers==4.44.2 + tokenizers (pinned)...



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


   📥 Installing biopython...



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


   ✅ biopython installed
   ✅ pandas already installed
   ✅ matplotlib already installed
   ✅ seaborn already installed
   ✅ tqdm already installed
   ✅ pyyaml already installed

   torch 2.4.1+cu124
   transformers 4.44.2

🖥️  GPU: NVIDIA H100 80GB HBM3 (85.0 GB)

✅ Setup complete!
⚠️  If you just upgraded transformers, RESTART THE KERNEL before continuing.


In [2]:
# ╔══════════════════════════════════════════════════════════╗
# ║                  SELECT DATABASE                         ║
# ╚══════════════════════════════════════════════════════════╝
# Note: single-filter mode is target-independent (operates only on
# the hydrolysis library and the 6 physicochemical filters of Step 2).
# Target selection from the standard NutrAI_Pipeline_v9 notebook is
# omitted here.

print("""
┌─────────── DATABASE ───────────┐
│  A = Foods                     │
│  B = Phytotherapeutics         │
│  C = Food by-products          │
│  G = ALL                       │
│                                │
│  Multiple: A,B  or  A,C        │
└────────────────────────────────┘
""")
DATABASE = input("DATABASE (A/B/C/G or combination e.g. A,B): ").strip().upper() or "B"

# ── Parse selection ──
DB_LETTER_MAP = {"A": ["alimenti"], "B": ["fitoterapici"], "C": ["scarti"], "G": ["alimenti", "fitoterapici", "scarti"]}

SELECTED_DBS = []
for letter in DATABASE.replace(" ", "").split(","):
    SELECTED_DBS.extend(DB_LETTER_MAP.get(letter, []))
SELECTED_DBS = list(dict.fromkeys(SELECTED_DBS))

DB_NAMES = {"alimenti": "🥦 Foods", "fitoterapici": "🌿 Phytotherapeutics", "scarti": "🗑️ By-products"}

print(f"""
{'='*50}
  DATABASE: {' + '.join(DB_NAMES.get(d, d) for d in SELECTED_DBS)}
  MODE:     Single-filter (target-independent)
{'='*50}
""")



┌─────────── DATABASE ───────────┐
│  A = Foods                     │
│  B = Phytotherapeutics         │
│  C = Food by-products          │
│  G = ALL                       │
│                                │
│  Multiple: A,B  or  A,C        │
└────────────────────────────────┘



DATABASE (A/B/C/G or combination e.g. A,B):  B



  DATABASE: 🌿 Phytotherapeutics
  MODE:     Single-filter (target-independent)



In [3]:
# ============================================================
# STEP 0c — ENVIRONMENT SETUP AND DATABASE LOADING
# ============================================================
import os, glob

# Environment
if os.path.exists('/workspace'):
    ENV = 'runpod'
    BASE_DIR = '/workspace'
    SAVE_DIR = '/workspace/nutrai_results'
elif os.path.exists('/content'):
    ENV = 'colab'
    BASE_DIR = '/content'
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        SAVE_DIR = '/content/drive/MyDrive/NutrAI_results'
    except:
        SAVE_DIR = None
else:
    ENV = 'local'
    BASE_DIR = os.path.expanduser('~')
    SAVE_DIR = os.path.join(BASE_DIR, 'nutrai_results')

print(f"🚀 {ENV}")

# Directories
WORK_DIR = os.path.join(BASE_DIR, "nutrai_pipeline")
DATA_DIR = os.path.join(WORK_DIR, "data")
RESULTS_DIR = os.path.join(WORK_DIR, "results")
BOLTZ_DIR = os.path.join(WORK_DIR, "boltz2_export")
DB_PATH = os.path.join(WORK_DIR, "nutrai.db")
for d in [WORK_DIR, DATA_DIR, RESULTS_DIR, BOLTZ_DIR]:
    os.makedirs(d, exist_ok=True)
if SAVE_DIR:
    os.makedirs(SAVE_DIR, exist_ok=True)

# ── Database map ──
DB_MAP = {
    "A": ["alimenti"], "B": ["fitoterapici"], "C": ["scarti"],
    "D": ["alimenti", "fitoterapici"], "E": ["alimenti", "scarti"],
    "F": ["fitoterapici", "scarti"], "G": ["alimenti", "fitoterapici", "scarti"],
}
DB_CATEGORIES_MAP = {
    "alimenti": ["alimenti_cereali", "alimenti_legumi", "alimenti_semi_oleosi", "alimenti_pseudocereali", "alimenti_microalghe"],
    "fitoterapici": ["fito_funghi", "fito_piante"],
    "scarti": ["scarti_bucce_frutta", "scarti_bucce_ortaggi", "scarti_gusci_semi"],
}
DB_NAMES = {"alimenti": "🥦 Foods", "fitoterapici": "🌿 Phytotherapeutics", "scarti": "🗑️ By-products"}

# ── Find and extract tar.gz ──
print("\n🔍 Searching for tar.gz databases...\n")
tar_files = set()
for search in [BASE_DIR, SAVE_DIR or "", WORK_DIR]:
    if search and os.path.exists(search):
        for tf in glob.glob(os.path.join(search, "db_*.tar.gz")):
            tar_files.add(tf)
        for tf in glob.glob(os.path.join(search, "**", "db_*.tar.gz"), recursive=True):
            tar_files.add(tf)

FOUND_DBS = []
for tf in sorted(tar_files):
    fname = os.path.basename(tf)
    os.system(f'tar xf "{tf}" -C {DATA_DIR} 2>/dev/null')
    os.system(f'tar xf "{tf}" -C / 2>/dev/null')
    for db_key in ["alimenti", "fitoterapici", "scarti", "TUTTI"]:
        if db_key in fname:
            FOUND_DBS.append(db_key)
            print(f"   ✅ {DB_NAMES.get(db_key, fname)}: {fname}")
            break

# ── Determine active categories ──
if DATABASE.upper() == "AUTO":
    # Auto-detect from found tar.gz
    if "TUTTI" in FOUND_DBS:
        SELECTED_DBS = ["alimenti", "fitoterapici", "scarti"]
    else:
        SELECTED_DBS = [d for d in FOUND_DBS if d != "TUTTI"]
else:
    SELECTED_DBS = DB_MAP.get(DATABASE.upper(), ["alimenti", "fitoterapici", "scarti"])

SELECTED_CATEGORIES = []
for db in SELECTED_DBS:
    SELECTED_CATEGORIES.extend(DB_CATEGORIES_MAP.get(db, []))

if not SELECTED_CATEGORIES:
    fasta_count = len(glob.glob(os.path.join(DATA_DIR, "*.fasta")))
    if fasta_count > 0:
        SELECTED_CATEGORIES = [cat for cats in DB_CATEGORIES_MAP.values() for cat in cats]
        print(f"   ℹ️ {fasta_count} fasta found — using all")
    else:
        raise FileNotFoundError("No database found! Upload tar.gz files to /workspace/")

# Single-filter mode: no target selection needed
SELECTED_TARGETS = []

# ── Summary ──
n_fasta = len(glob.glob(os.path.join(DATA_DIR, "*.fasta")))
print(f"\n{'='*50}")
print(f"  DATABASE: {' + '.join(DB_NAMES.get(d, d) for d in SELECTED_DBS)}")
print(f"  FASTA:    {n_fasta} files found")
print(f"{'='*50}")


🚀 runpod

🔍 Searching for tar.gz databases...

   ✅ 🥦 Foods: db_alimenti.tar.gz
   ✅ 🌿 Phytotherapeutics: db_fitoterapici.tar.gz
   ✅ 🗑️ By-products: db_scarti.tar.gz

  DATABASE: 🌿 Phytotherapeutics
  FASTA:    50 files found


## ⚙️ STEP 1 — Proteome Loading

In single-filter mode the configuration step is reduced to proteome loading only — no molecular target is needed since the 6 physicochemical filters of Step 2 are target-independent.

In [4]:
# ============================================================
# STEP 1 — Proteome loading config (target-independent in this mode)
# ============================================================
# In single-filter mode no molecular target is required.
# The TARGETS configuration block from the standard NutrAI_Pipeline_v9
# notebook is omitted here.

import os
import json

print("Single-filter mode — no target configuration required.")
print("Proceeding to proteome loading.")


Single-filter mode — no target configuration required.
Proceeding to proteome loading.


In [5]:
# ============================================================
# STEP 1b — PROTEOMES (from definitive PATCH, 3 databases)
# ============================================================

PROTEOMES = {
    "Triticum_aestivum": {
        "name": "Common wheat",
        "organism": "Triticum aestivum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000019116%29",
        "priority": "high",
        "category": "alimenti_cereali",
        "est_proteins": 130692,
        "proteome_id": "UP000019116",
        "note": "Reference proteome. Glutenins, gliadins, LTP."
    },
    "Oryza_sativa": {
        "name": "Rice",
        "organism": "Oryza sativa subsp. japonica",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000059680%29",
        "priority": "high",
        "category": "alimenti_cereali",
        "est_proteins": 48847,
        "proteome_id": "UP000059680",
        "note": "FIX: full proteome (was 4,197 with reviewed filter). Oryzanins, ACE-inhibitors."
    },
    "Zea_mays": {
        "name": "Maize",
        "organism": "Zea mays",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000007305%29",
        "priority": "medium",
        "category": "alimenti_cereali",
        "est_proteins": 63237,
        "proteome_id": "UP000007305",
        "note": "OK. Zeins, storage proteins."
    },
    "Hordeum_vulgare": {
        "name": "Barley",
        "organism": "Hordeum vulgare",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000011116%29",
        "priority": "medium",
        "category": "alimenti_cereali",
        "est_proteins": 35907,
        "proteome_id": "UP000011116",
        "note": "OK. Hordeins, lunasin-like."
    },
    "Avena_sativa": {
        "name": "Oat",
        "organism": "Avena sativa",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A4498%29",
        "priority": "medium",
        "category": "alimenti_cereali",
        "est_proteins": 95437,
        "proteome_id": "N/A (taxonomy query)",
        "note": "FIX: original proteome_id did not exist. Avenins, beta-glucanases."
    },
    "Glycine_max": {
        "name": "Soybean",
        "organism": "Glycine max",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000008827%29",
        "priority": "high",
        "category": "alimenti_legumi",
        "est_proteins": 74859,
        "proteome_id": "UP000008827",
        "note": "FIX: full proteome (was 439 with reviewed filter). Lunasin, BBI, glycinin."
    },
    "Cicer_arietinum": {
        "name": "Chickpea",
        "organism": "Cicer arietinum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3827%29",
        "priority": "high",
        "category": "alimenti_legumi",
        "est_proteins": 31239,
        "proteome_id": "UP000023160 (taxonomy query per completezza)",
        "note": "FIX: ref proteome had only 2,670. Taxonomy query gives 31,239."
    },
    "Pisum_sativum": {
        "name": "Pea",
        "organism": "Pisum sativum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000325105%29",
        "priority": "high",
        "category": "alimenti_legumi",
        "est_proteins": 3756,
        "proteome_id": "UP000325105",
        "note": "OK. Albumins, legumins."
    },
    "Arachis_hypogaea": {
        "name": "Peanut",
        "organism": "Arachis hypogaea",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3818%29",
        "priority": "high",
        "category": "alimenti_legumi",
        "est_proteins": 102020,
        "proteome_id": "UP000321265 (taxonomy query per completezza)",
        "note": "FIX: ref proteome had only 4,356. Taxonomy gives 102,020. Skin = by-product."
    },
    "Lupinus_albus": {
        "name": "Lupin",
        "organism": "Lupinus albus",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3870%29",
        "priority": "high",
        "category": "alimenti_legumi",
        "est_proteins": 38111,
        "proteome_id": "N/A (taxonomy query)",
        "note": "FIX: original proteome_id did not exist. Conglutins, hypoglycemic."
    },
    "Phaseolus_vulgaris": {
        "name": "Bean",
        "organism": "Phaseolus vulgaris",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000000226%29",
        "priority": "medium",
        "category": "alimenti_legumi",
        "est_proteins": 30854,
        "proteome_id": "UP000000226",
        "note": "OK. Phaseolins, lectins."
    },
    "Vigna_unguiculata": {
        "name": "Cowpea",
        "organism": "Vigna unguiculata",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3917%29",
        "priority": "medium",
        "category": "alimenti_legumi",
        "est_proteins": 39956,
        "proteome_id": "UP000234680 (taxonomy query per completezza)",
        "note": "FIX: ref proteome had only 1,698. Taxonomy gives 39,956."
    },
    "Helianthus_annuus": {
        "name": "Sunflower",
        "organism": "Helianthus annuus",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000215914%29",
        "priority": "medium",
        "category": "alimenti_semi_oleosi",
        "est_proteins": 88555,
        "proteome_id": "UP000215914",
        "note": "OK. SFTI-1 (natural cyclic peptide!)."
    },
    "Cannabis_sativa": {
        "name": "Hemp",
        "organism": "Cannabis sativa",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3483%29",
        "priority": "high",
        "category": "alimenti_semi_oleosi",
        "est_proteins": 79317,
        "proteome_id": "UP000563258 (taxonomy query per completezza)",
        "note": "FIX: ref proteome had only 5,308. Taxonomy gives 79,317. Edestin, 2S albumin."
    },
    "Sesamum_indicum": {
        "name": "Sesame",
        "organism": "Sesamum indicum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A4182%29",
        "priority": "medium",
        "category": "alimenti_semi_oleosi",
        "est_proteins": 29855,
        "proteome_id": "UP000260353 (taxonomy query per completezza)",
        "note": "FIX: ref proteome had only 4,420. Taxonomy gives 29,855."
    },
    "Cucurbita_maxima": {
        "name": "Pumpkin",
        "organism": "Cucurbita maxima",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3661%29",
        "priority": "low",
        "category": "alimenti_semi_oleosi",
        "est_proteins": 34748,
        "proteome_id": "UP000311284 (taxonomy query per completezza)",
        "note": "FIX: ref proteome had only 3,642. Taxonomy gives 34,748. Pumpkin seeds."
    },
    "Chenopodium_quinoa": {
        "name": "Quinoa",
        "organism": "Chenopodium quinoa",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A63459%29",
        "priority": "high",
        "category": "alimenti_pseudocereali",
        "est_proteins": 34188,
        "proteome_id": "N/A (taxonomy query)",
        "note": "FIX: original proteome_id did not exist. Chenopodin, saponins."
    },
    "Arthrospira_platensis": {
        "name": "Spirulina",
        "organism": "Arthrospira platensis",
        "url": "https://rest.uniprot.org/uniprotkb/stream?query=organism_id:118562&format=fasta",
        "priority": "high",
        "category": "alimenti_microalghe",
        "est_proteins": 4904,
        "proteome_id": "UP001547885",
        "note": "OK. C-phycocyanin, SYSQACHNR validated."
    },
    "Chlorella_vulgaris": {
        "name": "Chlorella",
        "organism": "Chlorella vulgaris",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3077%29",
        "priority": "high",
        "category": "alimenti_microalghe",
        "est_proteins": 9831,
        "proteome_id": "N/A (taxonomy query — vecchio ID era del pisello!)",
        "note": "CRITICAL FIX: proteome_id was UP000325105 = Pisum sativum! Corrected."
    },
    "Pleurotus_ostreatus": {
        "name": "Oyster mushroom",
        "organism": "Pleurotus ostreatus",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000623687%29",
        "priority": "high",
        "category": "fito_funghi",
        "est_proteins": 11624,
        "proteome_id": "UP000623687",
        "note": "OK. Pleurotin, lovastatin."
    },
    "Cordyceps_militaris": {
        "name": "Cordyceps",
        "organism": "Cordyceps militaris",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000001610%29",
        "priority": "high",
        "category": "fito_funghi",
        "est_proteins": 9651,
        "proteome_id": "UP000001610",
        "note": "OK. Cordycepin, performance, longevity."
    },
    "Ophiocordyceps_sinensis": {
        "name": "Cordyceps sinensis",
        "organism": "Ophiocordyceps sinensis",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000557566%29",
        "priority": "high",
        "category": "fito_funghi",
        "est_proteins": 9923,
        "proteome_id": "UP000557566",
        "note": "OK. Traditional caterpillar fungus."
    },
    "Grifola_frondosa": {
        "name": "Maitake",
        "organism": "Grifola frondosa",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000092993%29",
        "priority": "high",
        "category": "fito_funghi",
        "est_proteins": 14985,
        "proteome_id": "UP000092993",
        "note": "OK. Grifolan (beta-glucan), immunomodulation."
    },
    "Lentinula_edodes": {
        "name": "Shiitake",
        "organism": "Lentinula edodes",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000188533%29",
        "priority": "high",
        "category": "fito_funghi",
        "est_proteins": 12046,
        "proteome_id": "UP000188533",
        "note": "OK. Lentinan (beta-glucan)."
    },
    "Trametes_versicolor": {
        "name": "Turkey tail",
        "organism": "Trametes versicolor",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000054317%29",
        "priority": "medium",
        "category": "fito_funghi",
        "est_proteins": 1022,
        "proteome_id": "UP000054317",
        "note": "OK. PSK/PSP polysaccharides."
    },
    "Ganoderma_lucidum": {
        "name": "Reishi",
        "organism": "Ganoderma lucidum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A5315%29",
        "priority": "medium",
        "category": "fito_funghi",
        "est_proteins": 360,
        "proteome_id": "N/A",
        "note": "Limited proteome. Known triterpenes."
    },
    "Zingiber_officinale": {
        "name": "Ginger",
        "organism": "Zingiber officinale",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A94328%29",
        "priority": "high",
        "category": "fito_piante",
        "est_proteins": 71836,
        "proteome_id": "N/A",
        "note": "OK. Huge proteome. Known gingerols, unexplored peptides."
    },
    "Panax_ginseng": {
        "name": "Ginseng",
        "organism": "Panax ginseng",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A4054%29",
        "priority": "high",
        "category": "fito_piante",
        "est_proteins": 840,
        "proteome_id": "N/A",
        "note": "Limited proteome. Known ginsenosides. Adaptogen."
    },
    "Camellia_sinensis": {
        "name": "Green tea",
        "organism": "Camellia sinensis",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000306102%29",
        "priority": "high",
        "category": "fito_piante",
        "est_proteins": 30052,
        "proteome_id": "UP000306102",
        "note": "NEW. Known EGCG, unexplored peptides. TePigal."
    },
    "Artemisia_annua": {
        "name": "Sweet wormwood",
        "organism": "Artemisia annua",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000245207%29",
        "priority": "high",
        "category": "fito_piante",
        "est_proteins": 66067,
        "proteome_id": "UP000245207",
        "note": "NEW. Artemisinin, active antitumor."
    },
    "Olea_europaea": {
        "name": "Olive",
        "organism": "Olea europaea",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000594638%29",
        "priority": "high",
        "category": "fito_piante",
        "est_proteins": 78352,
        "proteome_id": "UP000594638",
        "note": "NEW. Oleuropein, hydroxytyrosol. Anti-inflammaging."
    },
    "Curcuma_longa": {
        "name": "Turmeric",
        "organism": "Curcuma longa",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A136217%29",
        "priority": "high",
        "category": "fito_piante",
        "est_proteins": 244,
        "proteome_id": "N/A",
        "note": "Limited proteome (244). Known curcuminoids."
    },
    "Glycyrrhiza_glabra": {
        "name": "Licorice",
        "organism": "Glycyrrhiza glabra",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A49827%29",
        "priority": "low",
        "category": "fito_piante",
        "est_proteins": 346,
        "proteome_id": "N/A",
        "note": "Limited proteome. Known glycyrrhizin."
    },
    "Moringa_oleifera": {
        "name": "Moringa",
        "organism": "Moringa oleifera",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3735%29",
        "priority": "medium",
        "category": "fito_piante",
        "est_proteins": 192,
        "proteome_id": "N/A",
        "note": "Limited proteome. Known antimicrobial peptides."
    },
    "Silybum_marianum": {
        "name": "Milk thistle",
        "organism": "Silybum marianum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A92921%29",
        "priority": "low",
        "category": "fito_piante",
        "est_proteins": 183,
        "proteome_id": "N/A",
        "note": "Limited proteome. Known silymarin. Hepatoprotective."
    },
    "Centella_asiatica": {
        "name": "Gotu kola",
        "organism": "Centella asiatica",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A48106%29",
        "priority": "low",
        "category": "fito_piante",
        "est_proteins": 170,
        "proteome_id": "N/A",
        "note": "Limited proteome. Neuroprotective, wound healing."
    },
    "Reynoutria_japonica": {
        "name": "Japanese knotweed",
        "organism": "Reynoutria japonica",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A488216%29",
        "priority": "low",
        "category": "fito_piante",
        "est_proteins": 174,
        "proteome_id": "N/A",
        "note": "Limited proteome. Main resveratrol source."
    },
    "Astragalus_membranaceus": {
        "name": "Astragalus",
        "organism": "Astragalus membranaceus",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A1862641%29",
        "priority": "low",
        "category": "fito_piante",
        "est_proteins": 76,
        "proteome_id": "N/A",
        "note": "Very limited proteome (76). Immunomodulation."
    },
    "Cinnamomum_verum": {
        "name": "Cinnamon",
        "organism": "Cinnamomum verum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?query=taxonomy_id:13424&format=fasta",
        "priority": "medium",
        "category": "fito_piante",
        "est_proteins": 172,
        "proteome_id": "N/A",
        "note": "Limited. Known cinnamaldehyde."
    },
    "Crocus_sativus": {
        "name": "Saffron",
        "organism": "Crocus sativus",
        "url": "https://rest.uniprot.org/uniprotkb/stream?query=taxonomy_id:82528&format=fasta",
        "priority": "medium",
        "category": "fito_piante",
        "est_proteins": 315,
        "proteome_id": "N/A",
        "note": "Limited. Known crocin/safranal."
    },
    "Vanilla_planifolia": {
        "name": "Vanilla",
        "organism": "Vanilla planifolia",
        "url": "https://rest.uniprot.org/uniprotkb/stream?query=proteome:UP000636800&format=fasta",
        "priority": "medium",
        "category": "fito_piante",
        "est_proteins": 51925,
        "proteome_id": "UP000636800",
        "note": "Known vanillin. Full proteome."
    },
    "Solanum_lycopersicum": {
        "name": "Tomato (skin)",
        "organism": "Solanum lycopersicum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000004994%29",
        "priority": "high",
        "category": "scarti_bucce_frutta",
        "est_proteins": 34663,
        "proteome_id": "UP000004994",
        "note": "FIX: full proteome (was 511 with reviewed filter). Defensins, PR proteins."
    },
    "Vitis_vinifera": {
        "name": "Grape/Vine (pomace)",
        "organism": "Vitis vinifera",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000009183%29",
        "priority": "high",
        "category": "scarti_bucce_frutta",
        "est_proteins": 29758,
        "proteome_id": "UP000009183",
        "note": "FIX: full proteome (was 187 with reviewed filter). Thaumatin-like, LTP."
    },
    "Solanum_tuberosum": {
        "name": "Potato (peel)",
        "organism": "Solanum tuberosum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000011115%29",
        "priority": "medium",
        "category": "scarti_bucce_ortaggi",
        "est_proteins": 53106,
        "proteome_id": "UP000011115",
        "note": "OK. Patatins (protease inhibitors). Peel rich in defensins."
    },
    "Punica_granatum": {
        "name": "Pomegranate (peel)",
        "organism": "Punica granatum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000233551%29",
        "priority": "high",
        "category": "scarti_bucce_frutta",
        "est_proteins": 50426,
        "proteome_id": "UP000233551",
        "note": "NEW. Peels very rich in proteins, antioxidants."
    },
    "Citrus_sinensis": {
        "name": "Orange (peel)",
        "organism": "Citrus sinensis",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000829398%29",
        "priority": "high",
        "category": "scarti_bucce_frutta",
        "est_proteins": 80120,
        "proteome_id": "UP000829398",
        "note": "NEW. Citrus peels, known flavonoids."
    },
    "Malus_domestica": {
        "name": "Apple (peel/core)",
        "organism": "Malus domestica",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000290289%29",
        "priority": "medium",
        "category": "scarti_bucce_frutta",
        "est_proteins": 42478,
        "proteome_id": "UP000290289",
        "note": "NEW. Peels and cores."
    },
    "Musa_acuminata": {
        "name": "Banana (peel)",
        "organism": "Musa acuminata",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000012960%29",
        "priority": "medium",
        "category": "scarti_bucce_frutta",
        "est_proteins": 40347,
        "proteome_id": "UP000012960",
        "note": "NEW. [REF] Banana peels."
    },
    "Persea_americana": {
        "name": "Avocado (seeds/peel)",
        "organism": "Persea americana",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP001234297%29",
        "priority": "medium",
        "category": "scarti_bucce_frutta",
        "est_proteins": 36165,
        "proteome_id": "UP001234297",
        "note": "NEW. [REF] Avocado seeds and peels."
    },
    "Ananas_comosus": {
        "name": "Pineapple (peel/core)",
        "organism": "Ananas comosus",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000515123%29",
        "priority": "medium",
        "category": "scarti_bucce_frutta",
        "est_proteins": 29712,
        "proteome_id": "UP000515123",
        "note": "NEW. Bromelain in core."
    },
    "Prunus_persica": {
        "name": "Peach (pit/skin)",
        "organism": "Prunus persica",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000006882%29",
        "priority": "low",
        "category": "scarti_bucce_frutta",
        "est_proteins": 38732,
        "proteome_id": "UP000006882",
        "note": "NEW. [REF] Pits and skins."
    },
    "Prunus_avium": {
        "name": "Cherry (pit)",
        "organism": "Prunus avium",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000515124%29",
        "priority": "low",
        "category": "scarti_bucce_frutta",
        "est_proteins": 30515,
        "proteome_id": "UP000515124",
        "note": "NEW. Cherry pits."
    },
    "Brassica_oleracea": {
        "name": "Broccoli (by-products)",
        "organism": "Brassica oleracea var. oleracea",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000032141%29",
        "priority": "high",
        "category": "scarti_bucce_ortaggi",
        "est_proteins": 58535,
        "proteome_id": "UP000032141",
        "note": "FIX: var. italica had 243. Using var. oleracea (same species). Sulforaphane."
    },
    "Daucus_carota": {
        "name": "Carrot (by-products)",
        "organism": "Daucus carota",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000077755%29",
        "priority": "medium",
        "category": "scarti_bucce_ortaggi",
        "est_proteins": 35531,
        "proteome_id": "UP000077755",
        "note": "NEW. [REF] Carrot by-products."
    },
    "Cucurbita_pepo": {
        "name": "Zucchini (by-products)",
        "organism": "Cucurbita pepo",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000504609%29",
        "priority": "low",
        "category": "scarti_bucce_ortaggi",
        "est_proteins": 35465,
        "proteome_id": "UP000504609",
        "note": "NEW. Peels and seeds."
    },
    "Beta_vulgaris": {
        "name": "Sugar beet (by-products)",
        "organism": "Beta vulgaris",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000035740%29",
        "priority": "low",
        "category": "scarti_bucce_ortaggi",
        "est_proteins": 7679,
        "proteome_id": "UP000035740",
        "note": "NEW. Known betalains."
    },
    "Capsicum_annuum": {
        "name": "Bell pepper (by-products)",
        "organism": "Capsicum annuum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000222542%29",
        "priority": "low",
        "category": "scarti_bucce_ortaggi",
        "est_proteins": 35548,
        "proteome_id": "UP000222542",
        "note": "NEW. [REF] Known capsaicin."
    },
    "Juglans_regia": {
        "name": "Walnut (shell/hull)",
        "organism": "Juglans regia",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000619265%29",
        "priority": "medium",
        "category": "scarti_gusci_semi",
        "est_proteins": 39594,
        "proteome_id": "UP000619265",
        "note": "NEW. Walnut shells and hulls."
    },
    "Prunus_dulcis": {
        "name": "Almond (shell)",
        "organism": "Prunus dulcis",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP001054821%29",
        "priority": "medium",
        "category": "scarti_gusci_semi",
        "est_proteins": 44957,
        "proteome_id": "UP001054821",
        "note": "NEW. [REF] Almond shells."
    },
    "Coffea_arabica": {
        "name": "Coffee (grounds)",
        "organism": "Coffea arabica",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP001652660%29",
        "priority": "high",
        "category": "scarti_gusci_semi",
        "est_proteins": 65937,
        "proteome_id": "UP001652660",
        "note": "NEW. Coffee grounds, huge waste volume."
    },
    "Theobroma_cacao": {
        "name": "Cocoa (shell)",
        "organism": "Theobroma cacao",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000026915%29",
        "priority": "medium",
        "category": "scarti_gusci_semi",
        "est_proteins": 40609,
        "proteome_id": "UP000026915",
        "note": "NEW. [REF] Cocoa shells."
    }
}

# Apply proteome selection (based on DATABASE chosen in cell 2)
active_proteomes = {}
for k, v in PROTEOMES.items():
    cat = v.get("category", "")
    if cat in SELECTED_CATEGORIES:
        active_proteomes[k] = v

if not active_proteomes:
    print("⚠️ No proteome matched — loading all")
    active_proteomes = PROTEOMES

n_active = len(active_proteomes)
n_total = len(PROTEOMES)
total_prot = sum(v.get("est_proteins", 0) for v in active_proteomes.values())
print(f"\n🌱 Active proteomes: {n_active} of {n_total}")
for k, v in active_proteomes.items():
    print(f"   {v['name']:25s} {v['category']:25s} ~{v.get('est_proteins',0):>8,}")
print(f"\n📦 Estimated total: {total_prot:,} proteins")



🌱 Active proteomes: 22 of 61
   Oyster mushroom           fito_funghi               ~  11,624
   Cordyceps                 fito_funghi               ~   9,651
   Cordyceps sinensis        fito_funghi               ~   9,923
   Maitake                   fito_funghi               ~  14,985
   Shiitake                  fito_funghi               ~  12,046
   Turkey tail               fito_funghi               ~   1,022
   Reishi                    fito_funghi               ~     360
   Ginger                    fito_piante               ~  71,836
   Ginseng                   fito_piante               ~     840
   Green tea                 fito_piante               ~  30,052
   Sweet wormwood            fito_piante               ~  66,067
   Olive                     fito_piante               ~  78,352
   Turmeric                  fito_piante               ~     244
   Licorice                  fito_piante               ~     346
   Moringa                   fito_piante               ~    

## 📂 STEP 2 — Load Proteomes
Load proteomes from downloaded databases (tar.gz). If missing, you can upload them.


In [6]:
# ============================================================
# STEP 2 — LOAD PROTEOMES
# ============================================================
import os, glob, subprocess

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

def parse_fasta(filepath):
    """FASTA parser. Yields (header, sequence)."""
    header, seq_parts = "", []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if header:
                    yield header, "".join(seq_parts)
                header = line[1:]
                seq_parts = []
            else:
                seq_parts.append(line.upper())
    if header:
        yield header, "".join(seq_parts)

# Search tar.gz EVERYWHERE in /workspace and /content
print("🔍 Searching for tar.gz databases...\n")
tar_files = []
for search in ["/workspace", "/content", BASE_DIR, WORK_DIR, SAVE_DIR or ""]:
    if search and os.path.exists(search):
        tar_files.extend(glob.glob(os.path.join(search, "db_*.tar.gz")))
        tar_files.extend(glob.glob(os.path.join(search, "**", "db_*.tar.gz"), recursive=True))

tar_files = list(set(tar_files))  # dedupe

if tar_files:
    for tf in tar_files:
        # Try direct extraction first
        ret = os.system(f'tar xf "{tf}" -C {DATA_DIR} 2>/dev/null')
        if ret != 0:
            # If it fails, try extracting into root (absolute paths in tar)
            os.system(f'tar xf "{tf}" -C / 2>/dev/null')
            # Then find extracted fasta files and copy them to DATA_DIR
            for fasta in glob.glob("/workspace/nutrai_pipeline/data/*.fasta"):
                if not os.path.exists(os.path.join(DATA_DIR, os.path.basename(fasta))):
                    os.system(f'cp "{fasta}" "{DATA_DIR}/"')
        print(f"   ✅ {os.path.basename(tf)}")
else:
    # No tar found — manual upload
    print("❌ No db_*.tar.gz found!")
    if ENV == 'colab':
        from google.colab import files
        print("📂 Select tar.gz files:")
        uploaded = files.upload()
        for fname in uploaded.keys():
            os.system(f'tar xf "{fname}" -C {DATA_DIR} 2>/dev/null')
            os.system(f'tar xf "{fname}" -C / 2>/dev/null')
            print(f"   ✅ {fname}")
    else:
        print(f"   Upload db_*.tar.gz files to /workspace/ and re-run this cell.")
        raise FileNotFoundError("No database found.")

# Count fasta files
proteome_stats = {}
for fpath in sorted(glob.glob(os.path.join(DATA_DIR, "*.fasta"))):
    pname = os.path.basename(fpath).replace(".fasta", "")
    if pname in active_proteomes:
        n_seq = sum(1 for line in open(fpath) if line.startswith(">"))
        if n_seq > 0:
            proteome_stats[pname] = n_seq
            name = active_proteomes[pname].get("name", pname)
            print(f"   ✅ {name:25s} {n_seq:>8,} proteins")
        else:
            print(f"   ⚠️ {pname}: empty file!")

missing = [k for k in active_proteomes if k not in proteome_stats]
if missing:
    print(f"\n⚠️ Missing {len(missing)} fasta:")
    for m in missing:
        print(f"   ❌ {active_proteomes[m]['name']} ({m})")

total_proteins = sum(proteome_stats.values())
print(f"\n📦 Total: {total_proteins:,} proteins from {len(proteome_stats)} proteomes")
if len(proteome_stats) > 0:
    print("✅ Ready!")
else:
    raise FileNotFoundError("No fasta loaded!")


🔍 Searching for tar.gz databases...

   ✅ db_scarti.tar.gz
   ✅ db_alimenti.tar.gz
   ✅ db_fitoterapici.tar.gz
   ✅ Sweet wormwood              66,067 proteins
   ✅ Astragalus                      76 proteins
   ✅ Green tea                   30,052 proteins
   ✅ Gotu kola                      170 proteins
   ✅ Cinnamon                       172 proteins
   ✅ Cordyceps                    9,651 proteins
   ✅ Saffron                        315 proteins
   ✅ Turmeric                       244 proteins
   ✅ Reishi                         360 proteins
   ✅ Licorice                       346 proteins
   ✅ Maitake                     14,985 proteins
   ✅ Shiitake                    12,046 proteins
   ✅ Moringa                        192 proteins
   ✅ Olive                       78,352 proteins
   ✅ Cordyceps sinensis           9,923 proteins
   ✅ Ginseng                        840 proteins
   ✅ Oyster mushroom             11,624 proteins
   ✅ Japanese knotweed              185 proteins
   ✅ Mi

## 🧪 STEP 3 — In Silico Enzymatic Hydrolysis
Instead of a brute sliding window, we simulate cleavage with **6 real enzymes**:
- **Trypsin**: cleaves after K, R
- **Pepsin**: cleaves after F, Y, W, L
- **Chymotrypsin**: cleaves after F, Y, W
- **Papain**: cleaves after R, K, Q, H, G, Y, F, W (broad)
- **Bromelain**: cleaves after K, R, A, Y
- **Alcalase**: cleaves after F, W, Y, L, I, V, M, A (broad)

This reduces fragments by **~97%** and generates only industrially producible peptides.


In [7]:
# ============================================================
# STEP 3 — IN SILICO ENZYMATIC HYDROLYSIS (6 ENZYMES)
# ============================================================
import csv
from tqdm.auto import tqdm
from collections import defaultdict

# Enzyme specificities: the enzyme cleaves AFTER these residues
ENZYMES = {
    "Trypsin":      set("KR"),
    "Pepsin":       set("FYWL"),
    "Chymotrypsin": set("FYW"),
    "Papain":       set("RKQHGYFW"),
    "Bromelain":    set("KRAY"),
    "Alcalase":     set("FWYLIVMA"),
}

# Parameters
MIN_LEN = 6
MAX_LEN = 20
AROMATIC = set("FWYH")
BASIC = set("RKH")

def enzymatic_digest(protein_seq, enzyme_cuts):
    """
    Simulate enzymatic cleavage of a protein.
    The enzyme cleaves AFTER the specified residues.
    Returns a list of (peptide, start_pos).
    """
    fragments = []
    start = 0
    for i, aa in enumerate(protein_seq):
        if aa in enzyme_cuts:
            frag = protein_seq[start:i+1]
            if MIN_LEN <= len(frag) <= MAX_LEN:
                fragments.append((frag, start))
            start = i + 1
    # Last fragment
    frag = protein_seq[start:]
    if MIN_LEN <= len(frag) <= MAX_LEN:
        fragments.append((frag, start))
    return fragments

def passes_biofilter(peptide):
    """Biochemical filter: at least 1 aromatic AND 1 basic, standard aa only."""
    if not all(aa in VALID_AA for aa in peptide):
        return False
    has_aromatic = any(aa in AROMATIC for aa in peptide)
    has_basic = any(aa in BASIC for aa in peptide)
    return has_aromatic and has_basic

# ── Processing ──
PEPTIDES_CSV = os.path.join(WORK_DIR, "peptides_hydrolysis.csv")
seen_peptides = set()
total_peptides = 0
stats_per_proteome = defaultdict(int)
stats_per_enzyme = defaultdict(int)

print("🧪 In silico enzymatic hydrolysis...\n")

with open(PEPTIDES_CSV, "w", newline="") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(["peptide", "length", "protein_id", "protein_name",
                     "source", "position", "enzyme"])

    for pname, pdata in active_proteomes.items():
        fpath = os.path.join(DATA_DIR, f"{pname}.fasta")
        if not os.path.exists(fpath):
            continue

        n_seq = proteome_stats.get(pname, 0)
        pep_count = 0

        for header, seq in tqdm(parse_fasta(fpath), total=n_seq,
                                desc=f"🧪 {pdata['name'][:20]:20s}"):
            clean = "".join(c for c in seq if c in VALID_AA)
            if len(clean) < MIN_LEN:
                continue

            prot_id = header.split("|")[1] if "|" in header else header.split()[0]
            prot_name = header.split("|")[2].split(" OS=")[0] if "|" in header and "OS=" in header else header[:80]

            # Digest with ALL enzymes
            for enz_name, enz_cuts in ENZYMES.items():
                for frag, start_pos in enzymatic_digest(clean, enz_cuts):
                    if frag in seen_peptides:
                        continue
                    if not passes_biofilter(frag):
                        continue
                    seen_peptides.add(frag)
                    writer.writerow([frag, len(frag), prot_id, prot_name,
                                    pname, f"{start_pos+1}-{start_pos+len(frag)}",
                                    enz_name])
                    total_peptides += 1
                    pep_count += 1
                    stats_per_enzyme[enz_name] += 1

        stats_per_proteome[pname] = pep_count

# Report
print(f"\n{'='*60}")
print(f"✅ Hydrolysis complete! {total_peptides:,} unique peptides generated")
print(f"\n📊 Per proteome:")
for pname, count in sorted(stats_per_proteome.items(), key=lambda x: -x[1]):
    print(f"   {active_proteomes[pname]['name']:25s} → {count:>8,} peptides")
print(f"\n🧪 Per enzyme:")
for enz, count in sorted(stats_per_enzyme.items(), key=lambda x: -x[1]):
    print(f"   {enz:15s} → {count:>8,} peptides")
print(f"\n💾 Saved to: {PEPTIDES_CSV}")


🧪 In silico enzymatic hydrolysis...



🧪 Oyster mushroom     :   0%|          | 0/11624 [00:00<?, ?it/s]

🧪 Cordyceps           :   0%|          | 0/9651 [00:00<?, ?it/s]

🧪 Cordyceps sinensis  :   0%|          | 0/9923 [00:00<?, ?it/s]

🧪 Maitake             :   0%|          | 0/14985 [00:00<?, ?it/s]

...

🧪 Astragalus          :   0%|          | 0/76 [00:00<?, ?it/s]

🧪 Cinnamon            :   0%|          | 0/172 [00:00<?, ?it/s]

🧪 Saffron             :   0%|          | 0/315 [00:00<?, ?it/s]

🧪 Vanilla             :   0%|          | 0/29005 [00:00<?, ?it/s]


✅ Hydrolysis complete! 8,645,509 unique peptides generated

📊 Per proteome:
   Sweet wormwood            → 1,656,100 peptides
   Ginger                    → 1,642,011 peptides
   Olive                     → 1,325,120 peptides
   Green tea                 → 1,026,235 peptides
   Vanilla                   →  696,181 peptides
   Oyster mushroom           →  501,490 peptides
   Maitake                   →  491,355 peptides
   Cordyceps                 →  461,584 peptides
   Shiitake                  →  408,823 peptides
   Cordyceps sinensis        →  363,077 peptides
   Turkey tail               →   30,614 peptides
   Ginseng                   →   11,961 peptides
   Reishi                    →   10,788 peptides
   Saffron                   →    5,481 peptides
   Licorice                  →    4,844 peptides
   Milk thistle              →    2,448 peptides
   Gotu kola                 →    2,140 peptides
   Moringa                   →    1,721 peptides
   Japanese knotweed         →    1,1

## STEP 3b — Single-Filter Mode (Table S2, SPAFE paper)

**Difference vs standard pipeline:**
- Standard (early-exit): peptide discarded at first failed filter → not tested on the others.
- Single-filter mode: each filter applied independently on the total → 6 standalone numbers.

Output: `single_filter_counts.json` + `single_filter_counts.txt` to forward to Minoo for Table S2.

No ESM-2, no GPU, no Boltz. Only statistical counts on the `peptides_hydrolysis.csv` just generated by STEP 3.

In [8]:
# ============================================================
# STEP 3b — SINGLE-FILTER MODE (Table S2)
# Count peptides passing EACH filter applied INDEPENDENTLY.
# No early-exit. No intermediate combinations.
# ============================================================
import csv, os, json
from tqdm.auto import tqdm

# ── Physicochemical constants (identical to STEP 3b standard pipeline v9) ──
# Radzicka & Wolfenden 1988: ΔG cyclohexane → water transfer (kcal/mol)
BOMAN_TRANSFER = {
    "A": 0.50, "R": -2.53, "N": -6.64, "D": -8.72, "C": -1.29,
    "Q": -5.54, "E": -6.81, "G": 1.15, "H": -4.66, "I": 4.92,
    "K": -5.55, "L": 4.92, "M": 2.35, "F": 2.98, "P": -0.02,
    "S": -3.40, "T": -2.57, "W": 2.33, "Y": -0.14, "V": 4.04,
}
CHARGE_AA = {"R": 1, "K": 1, "H": 0.5, "D": -1, "E": -1}
HYDRO_KD = {"A":1.8,"R":-4.5,"N":-3.5,"D":-3.5,"C":2.5,"Q":-3.5,"E":-3.5,"G":-0.4,
            "H":-3.2,"I":4.5,"L":3.8,"K":-3.9,"M":1.9,"F":2.8,"P":-1.6,"S":-0.8,
            "T":-0.7,"W":-0.9,"Y":-1.3,"V":4.2}
MW_AA = {"A":89.1,"R":174.2,"N":132.1,"D":133.1,"C":121.2,"Q":146.2,"E":147.1,"G":75.0,
         "H":155.2,"I":131.2,"L":131.2,"K":146.2,"M":149.2,"F":165.2,"P":115.1,"S":105.1,
         "T":119.1,"W":204.2,"Y":181.2,"V":117.1}

# Instability Index DIWV (Guruprasad 1990)
DIWV = {"WW":1,"WC":1,"WM":24.68,"WH":24.68,"WY":1,"WF":1,"WQ":1,"WN":13.34,
        "WE":1,"WD":-14.03,"WK":1,"WR":-14.03,"WS":1,"WT":-14.03,"WG":-9.37,
        "WA":-14.03,"WV":-7.49,"WL":13.34,"WI":1,"WP":-1.88,
        "CW":24.68,"CC":1,"CM":33.6,"CH":33.6,"CY":1,"CF":-14.03,"CQ":-6.54,
        "CN":1,"CE":1,"CD":20.26,"CK":1,"CR":1,"CS":1,"CT":33.6,"CG":1,
        "CA":1,"CV":-6.54,"CL":20.26,"CI":1,"CP":20.26,
        "AW":1,"AC":44.94,"AM":1,"AH":-7.49,"AY":1,"AF":1,"AQ":1,"AN":1,
        "AE":1,"AD":-7.49,"AK":1,"AR":1,"AS":1,"AT":1,"AG":1,
        "AA":1,"AV":1,"AL":1,"AI":1,"AP":20.26,
        "GW":13.34,"GC":1,"GM":1,"GH":1,"GY":-7.49,"GF":1,"GQ":1,"GN":-7.49,
        "GE":-6.54,"GD":1,"GK":-7.49,"GR":1,"GS":1,"GT":-7.49,"GG":1,
        "GA":1,"GV":1,"GL":1,"GI":-7.49,"GP":1}

def boman_index(seq):
    total = sum(BOMAN_TRANSFER.get(aa, 0) for aa in seq)
    return -total / len(seq) if seq else 0

def net_charge(seq):
    return sum(CHARGE_AA.get(aa, 0) for aa in seq)

def gravy(seq):
    vals = [HYDRO_KD.get(aa, 0) for aa in seq]
    return sum(vals) / len(vals) if vals else 0

def mol_weight(seq):
    return sum(MW_AA.get(aa, 0) for aa in seq) - (len(seq) - 1) * 18.02

def instability_index(seq):
    n = len(seq)
    if n < 2:
        return 100
    total = 0
    for j in range(n - 1):
        dipep = seq[j] + seq[j+1]
        total += DIWV.get(dipep, 1)
    return (10.0 / n) * total

def hydrophobic_ratio(seq):
    HYDROPHOBIC = set("AILMFPWV")
    return sum(1 for aa in seq if aa in HYDROPHOBIC) / len(seq) if seq else 0

# ── Final thresholds (identical to STEP 3b standard pipeline v9) ──
MIN_BOMAN = 1.0
MIN_GRAVY, MAX_GRAVY = -2.0, 0.5
MIN_CHARGE, MAX_CHARGE = -2, 4
MAX_INSTAB = 40
MAX_MW = 1500
MIN_HYDRO, MAX_HYDRO = 0.30, 0.60

# ── Read CSV ──
print(f"📂 Reading {PEPTIDES_CSV}...")
with open(PEPTIDES_CSV, "r") as f:
    all_peptides = [row["peptide"] for row in csv.DictReader(f)]

total = len(all_peptides)
print(f"   Total peptides: {total:,}\n")

# ── Single-filter counting (no early-exit) ──
counts = {"boman": 0, "gravy": 0, "charge": 0, "instab": 0, "mw": 0, "hydro": 0}

print("🔬 Computing standalone counts (single-filter mode)...")
for seq in tqdm(all_peptides):
    # Each filter applied INDEPENDENTLY
    if boman_index(seq) >= MIN_BOMAN:
        counts["boman"] += 1
    
    g = gravy(seq)
    if MIN_GRAVY <= g <= MAX_GRAVY:
        counts["gravy"] += 1
    
    ch = net_charge(seq)
    if MIN_CHARGE <= ch <= MAX_CHARGE:
        counts["charge"] += 1
    
    if instability_index(seq) < MAX_INSTAB:
        counts["instab"] += 1
    
    if mol_weight(seq) <= MAX_MW:
        counts["mw"] += 1
    
    hr = hydrophobic_ratio(seq)
    if MIN_HYDRO <= hr <= MAX_HYDRO:
        counts["hydro"] += 1

# ── AND-combined (sanity check vs standard pipeline) ──
print("\n🔁 AND-combined (sanity check)...")
passed_all = 0
for seq in tqdm(all_peptides):
    if boman_index(seq) < MIN_BOMAN: continue
    g = gravy(seq)
    if g < MIN_GRAVY or g > MAX_GRAVY: continue
    ch = net_charge(seq)
    if ch < MIN_CHARGE or ch > MAX_CHARGE: continue
    if instability_index(seq) >= MAX_INSTAB: continue
    if mol_weight(seq) > MAX_MW: continue
    hr = hydrophobic_ratio(seq)
    if hr < MIN_HYDRO or hr > MAX_HYDRO: continue
    passed_all += 1

# ── Report ──
print(f"\n{'='*60}")
print(f"  SINGLE-FILTER COUNTS — Table S2 (SPAFE paper)")
print(f"{'='*60}")
print(f"  {'Filter':<28} {'Pass (n)':>12} {'Pass %':>10}")
print(f"  {'-'*58}")
labels = {
    "boman":  f"Boman >= {MIN_BOMAN}",
    "gravy":  f"GRAVY [{MIN_GRAVY},{MAX_GRAVY}]",
    "charge": f"Charge [{MIN_CHARGE},{MAX_CHARGE}]",
    "instab": f"Instab. < {MAX_INSTAB}",
    "mw":     f"MW <= {MAX_MW}",
    "hydro":  f"Hydrophob. [{MIN_HYDRO},{MAX_HYDRO}]",
}
for key, lbl in labels.items():
    n = counts[key]
    pct = 100 * n / total
    print(f"  {lbl:<28} {n:>12,} {pct:>9.2f}%")
print(f"  {'-'*58}")
print(f"  {'TOTAL input':<28} {total:>12,}     100.00%")
print(f"  {'Combined AND (all 6)':<28} {passed_all:>12,} {100*passed_all/total:>9.2f}%")
print(f"{'='*60}")

# ── Save output for Minoo ──
OUT_JSON = os.path.join(WORK_DIR, "single_filter_counts.json")
OUT_TXT = os.path.join(WORK_DIR, "single_filter_counts.txt")

result = {
    "total_input": total,
    "standalone_pass_counts": counts,
    "combined_pass_count": passed_all,
    "thresholds": {
        "boman_min": MIN_BOMAN,
        "gravy_min": MIN_GRAVY, "gravy_max": MAX_GRAVY,
        "charge_min": MIN_CHARGE, "charge_max": MAX_CHARGE,
        "instab_max": MAX_INSTAB,
        "mw_max": MAX_MW,
        "hydro_min": MIN_HYDRO, "hydro_max": MAX_HYDRO,
    },
    "source_csv": PEPTIDES_CSV,
    "method": "Single-filter mode: each filter applied independently on the raw input (no early-exit). Last row is the AND-combined count (replicates standard pipeline STEP 3b)."
}
with open(OUT_JSON, "w") as f:
    json.dump(result, f, indent=2)
with open(OUT_TXT, "w") as f:
    f.write(f"Single-filter mode counts — NutrAI Pipeline Table S2\n")
    f.write(f"Source: {PEPTIDES_CSV}\n")
    f.write(f"Total input: {total:,}\n\n")
    for key, lbl in labels.items():
        n = counts[key]
        pct = 100 * n / total
        f.write(f"{lbl:<28} {n:>12,} ({pct:.2f}%)\n")
    f.write(f"\nCombined AND (all 6): {passed_all:,} ({100*passed_all/total:.2f}%)\n")

print(f"\n💾 Output saved:")
print(f"   {OUT_JSON}")
print(f"   {OUT_TXT}")
print(f"\n✅ data for Table S2.")


📂 Reading /workspace/nutrai_pipeline/peptides_hydrolysis.csv...
   Total peptides: 8,645,509

🔬 Computing standalone counts (single-filter mode)...


  0%|          | 0/8645509 [00:00<?, ?it/s]


🔁 AND-combined (sanity check)...


  0%|          | 0/8645509 [00:00<?, ?it/s]


  SINGLE-FILTER COUNTS — Table S2 (SPAFE paper)
  Filter                           Pass (n)     Pass %
  ----------------------------------------------------------
  Boman >= 1.0                    5,127,842     59.31%
  GRAVY [-2.0,0.5]                6,870,519     79.47%
  Charge [-2,4]                   8,021,155     92.78%
  Instab. < 40                    8,448,408     97.72%
  MW <= 1500                      5,959,152     68.93%
  Hydrophob. [0.3,0.6]            5,559,756     64.31%
  ----------------------------------------------------------
  TOTAL input                     8,645,509     100.00%
  Combined AND (all 6)            1,642,848     19.00%

💾 Output saved:
   /workspace/nutrai_pipeline/single_filter_counts.json
   /workspace/nutrai_pipeline/single_filter_counts.txt

✅ data for Table S2.
